# ESG KPI Extraction & Reconciliation — Controlled Benchmark

This notebook demonstrates the controlled benchmark workflow using
synthetic sustainability disclosures and hidden ground truth.

The benchmark follows the project design:

**hidden truth → synthetic disclosures → independent extraction methods
→ normalized predictions → benchmark evaluation**

It also demonstrates the reconciled hybrid workflow for controlled
missing-value and conflicting-disclosure cases.

All reported metrics below are calculated from actual extraction
outputs. No performance values are predetermined.


In [ ]:
import logging
import os
from pathlib import Path
from unittest.mock import patch

import pandas as pd
from IPython.display import display

from esg.benchmark.case_evaluation import (
    evaluate_benchmark_case,
)
from esg.benchmark.cases import load_benchmark_cases
from esg.benchmark.generator import generate_benchmark_pdfs
from esg.benchmark.truth import load_benchmark_truth
from esg.benchmark.workflow_case_evaluation import (
    evaluate_benchmark_workflow_case,
)
from esg.config import load_config


logging.getLogger("esg").setLevel(logging.WARNING)

# Independent LLM benchmark calls are opt-in because they require
# network access and may incur API usage.
RUN_LLM = False


In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in (current, *current.parents):
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "src" / "esg").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate the esg-llm-platform project root."
    )


ROOT = find_project_root()

TRUTH_PATH = (
    ROOT
    / "data"
    / "benchmark"
    / "truth"
    / "benchmark_truth.yaml"
)
CASES_PATH = (
    ROOT
    / "data"
    / "benchmark"
    / "cases"
    / "benchmark_cases.yaml"
)
GENERATED_DIR = (
    ROOT
    / "data"
    / "benchmark"
    / "generated"
)

truth = load_benchmark_truth(TRUTH_PATH)
cases = load_benchmark_cases(CASES_PATH)
kpi_schema = load_config().universal_kpis

generated = generate_benchmark_pdfs(
    TRUTH_PATH,
    CASES_PATH,
    GENERATED_DIR,
)

pdf_by_case = {
    path.stem: path
    for path in generated
}

print("Project root:", ROOT)
print("Benchmark cases:", len(cases))
print("Generated PDFs:", len(generated))
print("Schema metrics:", len(kpi_schema))
print("RUN_LLM:", RUN_LLM)


In [ ]:
case_rows = []

for case in cases:
    expected_reconciliation = (
        case.get("expected_reconciliation") or {}
    )

    case_rows.append(
        {
            "case_id": case["case_id"],
            "company_id": case["company_id"],
            "omitted_metrics": ", ".join(
                case.get("omitted_metrics") or []
            ),
            "conflicting_metrics": ", ".join(
                (case.get("conflicting_values") or {}).keys()
            ),
            "workflow_expectations": ", ".join(
                expected_reconciliation.keys()
            ),
        }
    )

case_inventory = pd.DataFrame(case_rows)

display(case_inventory)


## Independent method-level benchmark

Each extraction method is run independently. The runner has no access
to hidden truth and performs no fusion or reconciliation.

Hidden truth is supplied only after extraction to the benchmark
evaluator.

The LLM method is excluded by default. Set `RUN_LLM = True` explicitly
to include it.


In [ ]:
methods = [
    "table_grid",
    "table_plain",
    "regex",
    "nlp",
]

if RUN_LLM:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError(
            "RUN_LLM=True but OPENAI_API_KEY is not configured."
        )

    methods.append("llm")


method_records = []
case_method_results = {}

for case in cases:
    case_id = str(case["case_id"])
    pdf_path = pdf_by_case[case_id]

    for method in methods:
        result = evaluate_benchmark_case(
            str(pdf_path),
            case=case,
            truth=truth,
            method=method,
            kpi_schema=kpi_schema,
        )

        case_method_results[
            (case_id, method)
        ] = result

        summary = result["evaluation"]["summary"]

        method_records.append(
            {
                "case_id": case_id,
                "method": method,
                **summary,
            }
        )

method_results = pd.DataFrame(method_records)

metric_columns = [
    "detection_precision",
    "detection_recall",
    "numeric_value_accuracy",
    "unit_accuracy",
    "reporting_year_accuracy",
    "location_accuracy",
    "missing_value_accuracy",
    "extraction_coverage",
]

display(
    method_results[
        ["case_id", "method", *metric_columns]
    ].round(3)
)


In [ ]:
# Per-case comparison avoids hiding case-specific failures behind
# a single aggregate score.

for metric in metric_columns:
    print()
    print(metric)

    comparison = method_results.pivot(
        index="case_id",
        columns="method",
        values=metric,
    )

    display(comparison.round(3))


## Controlled missing-value case

`alpha_missing_water_consumption` deliberately omits
`water_consumption`.

This section shows whether each independent method correctly leaves the
metric unreported rather than assigning another water value to it.


In [ ]:
missing_case_id = "alpha_missing_water_consumption"

missing_rows = []

for method in methods:
    result = case_method_results[
        (missing_case_id, method)
    ]

    metric_result = result["evaluation"]["metrics"][
        "water_consumption"
    ]

    prediction = result["predictions"].get(
        "water_consumption",
        {},
    )

    missing_rows.append(
        {
            "method": method,
            "predicted_present": (
                metric_result["predicted_present"]
            ),
            "predicted_value": prediction.get("value"),
            "predicted_unit": prediction.get("unit"),
            "missing_value_correct": (
                metric_result["missing_value_correct"]
            ),
        }
    )

missing_results = pd.DataFrame(missing_rows)

display(missing_results)


## Reconciled hybrid workflow

The workflow benchmark evaluates final reconciliation decisions for
cases with explicit reconciliation expectations.

For reproducibility, LLM backfill is mocked to return no predictions in
this section. This ensures that the demonstration tests deterministic
evidence preservation and reconciliation rather than a stochastic
external API response.


In [ ]:
workflow_cases = [
    case
    for case in cases
    if case.get("expected_reconciliation")
]

workflow_rows = []

with patch(
    "esg.pipeline.pipeline.extract_kpis_llm",
    return_value={},
):
    for case in workflow_cases:
        case_id = str(case["case_id"])

        result = evaluate_benchmark_workflow_case(
            str(pdf_by_case[case_id]),
            case=case,
        )

        evaluation = result["evaluation"]

        actual_by_metric = {
            item.metric: item
            for item in result["results"]
        }

        for metric, metric_eval in (
            evaluation["metrics"].items()
        ):
            actual = actual_by_metric.get(metric)

            actual_status = None
            if actual is not None:
                actual_status = getattr(
                    actual.status,
                    "value",
                    actual.status,
                )

            expected_status = (
                case["expected_reconciliation"]
                [metric]
                .get("status")
            )

            workflow_rows.append(
                {
                    "case_id": case_id,
                    "metric": metric,
                    "expected_status": expected_status,
                    "actual_status": actual_status,
                    "expected_conflict_flag": (
                        metric_eval[
                            "expected_conflict_flag"
                        ]
                    ),
                    "actual_conflict_flag": (
                        metric_eval[
                            "actual_conflict_flag"
                        ]
                    ),
                    "conflict_correct": (
                        metric_eval["conflict_correct"]
                    ),
                    "expected_review_required": (
                        metric_eval[
                            "expected_review_required"
                        ]
                    ),
                    "actual_review_required": (
                        metric_eval[
                            "actual_review_required"
                        ]
                    ),
                    "review_correct": (
                        metric_eval["review_correct"]
                    ),
                    "conflict_detection_accuracy": (
                        evaluation["summary"][
                            "conflict_detection_accuracy"
                        ]
                    ),
                    "review_flag_accuracy": (
                        evaluation["summary"][
                            "review_flag_accuracy"
                        ]
                    ),
                }
            )

workflow_results = pd.DataFrame(workflow_rows)

display(workflow_results)


## Interpretation notes

- The benchmark metrics above are calculated from current extraction
  outputs against controlled hidden ground truth.
- Heuristic extractor confidence scores are **not** treated as measures
  of correctness.
- A value of `0.0` for a benchmark dimension can also reflect the
  evaluator's zero-denominator convention when that dimension has no
  applicable comparison in a particular case.
- Independent LLM benchmarking is opt-in because it requires external
  API access.
- The reconciled workflow demonstration disables LLM backfill so that
  conflict and review behavior is reproducible.
- Generated benchmark PDFs are derived artifacts and are intentionally
  excluded from Git.
